# Ingeniería de Características y Construcción del Dataset Final

## 🎯 Objetivo del Cuaderno
El propósito de este notebook es transformar un dataset transaccional plano en un conjunto de datos altamente predictivo mediante el uso de **Graph Data Science (GDS)** en Neo4j. El foco principal es extraer variables topológicas y de centralidad que revelen comportamientos sospechosos complejos (como triangulación de dinero y redes de colusión) que los modelos tabulares tradicionales no logran detectar de forma nativa.

---

## 🛡️ Prevención Estricta de Fuga de Datos (*Data Leakage*)
Trabajar con grafos y series temporales al mismo tiempo introduce un riesgo crítico: el sesgo de anticipación (*Look-ahead bias*). Si calculamos métricas de red utilizando transacciones del "futuro", el modelo de Machine Learning será artificialmente perfecto en el entrenamiento, pero fallará catastróficamente en producción. 

Para evitar esto, implementamos una infraestructura **Point-in-Time (PIT)** basada en dos estrategias:
1. **Grados Históricos Acumulados:** Filtros estrictos basados en la línea temporal de cada transacción individual.
2. **Algoritmos por Bloques Temporales Deslizantes:** Procesamiento iterativo en ventanas de tiempo cerradas para aislar el pasado del futuro en algoritmos globales como *PageRank* y *Louvain*.

---

## 1. Inicialización del Entorno y Carga de Datos

En esta celda preparamos el entorno de ejecución, validamos la infraestructura local y realizamos la primera lectura del dataset original. 

In [1]:
import os
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
import pandas as pd
import time
import csv
import gzip

load_dotenv(find_dotenv())

URI = os.getenv("NEO4J_URI")
AUTH = (os.getenv("NEO4J_USER"), os.getenv("NEO4J_PASSWORD"))


In [2]:

try:
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        driver.verify_connectivity()
        print("¡Conexión exitosa a Neo4j! 🚀")
except Exception as e:
    print(f"Error al conectar: {e}")

¡Conexión exitosa a Neo4j! 🚀


In [3]:
df = pd.read_csv('../data/raw/ml_dataset.csv.gz', compression='gzip')
df

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.00,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.00,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.00,0.00,0,0
...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.00,C776919290,0.00,339682.13,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.00,C1881841831,0.00,0.00,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.00,C1365125890,68488.84,6379898.11,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.00,C2080388513,0.00,0.00,1,0


## 2. Modelado e Ingesta de Datos en Grafo (Neo4j)

Para capturar los patrones complejos de fraude (como mulas de dinero o redes de lavado), modelamos nuestras transacciones tabulares como un grafo estructurado de la siguiente manera:
* **Nodos (`Account`):** Representan las cuentas bancarias (tanto de origen como de destino).
* **Relaciones (`TRANSACTION`):** Representan el flujo de dinero entre dos cuentas, almacenando los metadatos financieros (monto, balances, si es fraude, etc.) como propiedades de la relación.

**Estrategia de Optimización:**
Para ingestar más de 6.3 millones de filas sin desbordar la memoria RAM ni causar bloqueos en la base de datos, implementamos una carga por lotes (*batching*). El proceso se divide en:
1. Creación de índices y restricciones de unicidad.
2. Extracción e inyección de nodos únicos (evitando redundancia).
3. Inyección de relaciones usando `UNWIND` en Cypher para un procesamiento masivo eficiente.

In [4]:
def ingestar_raw_data(df, batch_size=50000):

    query_nodos_origen = """
    UNWIND $rows AS row
    MERGE (:Account {id: row.nameOrig})
    """
    
    query_nodos_destino = """
    UNWIND $rows AS row
    MERGE (:Account {id: row.nameDest})
    """
    
    query_relaciones = """
    UNWIND $rows AS row
    MATCH (orig:Account {id: row.nameOrig})
    MATCH (dest:Account {id: row.nameDest})
    CREATE (orig)-[:TRANSACTION {
        amount: toFloat(row.amount),
        type: row.type,
        step: toInteger(row.step),
        oldbalanceOrg: toFloat(row.oldbalanceOrg),
        newbalanceOrig: toFloat(row.newbalanceOrig),
        oldbalanceDest: toFloat(row.oldbalanceDest),
        newbalanceDest: toFloat(row.newbalanceDest),
        isFraud: toInteger(row.isFraud)
    }]->(dest)
    """

    total_filas = len(df)
    print(f"Preparando ingesta masiva de {total_filas} registros...")
    
    start_time = time.time()
    
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:
            
            print("[Fase 0] Creando restricciones de unicidad...")
            session.run("CREATE CONSTRAINT ACCOUNT_ID_UNIQUE IF NOT EXISTS FOR (a:Account) REQUIRE a.id IS UNIQUE")
            
            print("\n[Fase 1] Cargando nodos Account únicos a la base de datos...")
            
            origenes_unicos = df[['nameOrig']].drop_duplicates()
            destinos_unicos = df[['nameDest']].drop_duplicates()
            
            for i in range(0, len(origenes_unicos), batch_size):
                chunk = origenes_unicos.iloc[i:i + batch_size]
                batch = chunk.to_dict('records')
                session.execute_write(lambda tx: tx.run(query_nodos_origen, rows=batch))
            
            for i in range(0, len(destinos_unicos), batch_size):
                chunk = destinos_unicos.iloc[i:i + batch_size]
                batch = chunk.to_dict('records')
                session.execute_write(lambda tx: tx.run(query_nodos_destino, rows=batch))
                
            print(f"-> Nodos asegurados en el grafo. Tiempo: {round(time.time() - start_time, 2)} s")
            
            print("\n[Fase 2] Creando las 6M+ relaciones TRANSACTION (Carga por lotes)...")
            rel_start_time = time.time()
            
            for i in range(0, total_filas, batch_size):
                chunk = df.iloc[i:i + batch_size]
                
                batch_records = chunk[[
                    'nameOrig', 'nameDest', 'amount', 'type', 'step', 
                    'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud'
                ]].to_dict('records')
                
                session.execute_write(lambda tx: tx.run(query_relaciones, rows=batch_records))
                
                if i % 500000 == 0 and i > 0:
                    elapsed = time.time() - rel_start_time
                    print(f"\r   Progreso: {i:,}/{total_filas:,} relaciones inyectadas... [{elapsed:.2f} s]",end="",flush=True)
            
            end_time = time.time()
            print(f"\n--- INGESTA COMPLETA ---")
            print(f"Tiempo total del proceso: {round((end_time - start_time)/60, 2)} minutos.")

ingestar_raw_data(df)

Preparando ingesta masiva de 6362620 registros...
[Fase 0] Creando restricciones de unicidad...

[Fase 1] Cargando nodos Account únicos a la base de datos...
-> Nodos asegurados en el grafo. Tiempo: 191.06 s

[Fase 2] Creando las 6M+ relaciones TRANSACTION (Carga por lotes)...
   Progreso: 6,000,000/6,362,620 relaciones inyectadas... [356.34 s]
--- INGESTA COMPLETA ---
Tiempo total del proceso: 9.44 minutos.


## 3. Verificación de la ingesta y muestra de datos

Una vez finalizado el proceso de ingesta de datos, ejecutamos un análisis rápido de integridad estructural en Neo4j. El objetivo es validar que el volumen de entidades cargadas coincida con los archivos origen y verificar que las propiedades clave se hayan mapeado correctamente.

In [5]:
def verificar_ingesta_y_muestra():
    print("=== EJECUTANDO SANITY CHECK EN NEO4J ===")
    
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:
            total_nodos = session.run("MATCH (a:Account) RETURN count(a) AS total").single()["total"]
            
            total_relaciones = session.run("MATCH (:Account)-[t:TRANSACTION]->(:Account) RETURN count(t) AS total").single()["total"]
            
            print(f"✅ Nodos 'Account' cargados correctamente: {total_nodos:,}")
            print(f"✅ Relaciones 'TRANSACTION' creadas correctamente: {total_relaciones:,}\n")
            
            query_muestra = """
            MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account)
            RETURN 
                t.step AS Step,
                orig.id AS Cuenta_Origen,
                dest.id AS Cuenta_Destino,
                t.type AS Tipo_Tx,
                t.amount AS Monto,
                t.oldbalanceOrg AS Balance_Previo_Orig,
                t.isFraud AS Es_Fraude
            LIMIT 5
            """
            
            result = session.run(query_muestra)

            print("📊 Muestra de transacciones:\n")
            for record in result:
                print(record)

verificar_ingesta_y_muestra()

=== EJECUTANDO SANITY CHECK EN NEO4J ===
✅ Nodos 'Account' cargados correctamente: 9,073,900
✅ Relaciones 'TRANSACTION' creadas correctamente: 6,362,620

📊 Muestra de transacciones:

<Record Step=1 Cuenta_Origen='C1231006815' Cuenta_Destino='M1979787155' Tipo_Tx='PAYMENT' Monto=9839.64 Balance_Previo_Orig=170136.0 Es_Fraude=0>
<Record Step=1 Cuenta_Origen='C1666544295' Cuenta_Destino='M2044282225' Tipo_Tx='PAYMENT' Monto=1864.28 Balance_Previo_Orig=21249.0 Es_Fraude=0>
<Record Step=1 Cuenta_Origen='C1305486145' Cuenta_Destino='C553264065' Tipo_Tx='TRANSFER' Monto=181.0 Balance_Previo_Orig=181.0 Es_Fraude=1>
<Record Step=1 Cuenta_Origen='C840083671' Cuenta_Destino='C38997010' Tipo_Tx='CASH_OUT' Monto=181.0 Balance_Previo_Orig=181.0 Es_Fraude=1>
<Record Step=1 Cuenta_Origen='C2048537720' Cuenta_Destino='M1230701703' Tipo_Tx='PAYMENT' Monto=11668.14 Balance_Previo_Orig=41554.0 Es_Fraude=0>


## 4. Cálculo de Grados Históricos Point-in-Time (PIT)

En esta celda calculamos de forma nativa el **grado de salida** (`out_degree_hist`) de la cuenta origen y el **grado de entrada** (`in_degree_hist`) de la cuenta destino para cada transacción. Estas métricas capturan la acumulación histórica de actividad y el volumen de conexiones en la red.

### 🛡️ Prevención de Fuga de Datos (Data Leakage)
Al tratarse de un problema de series temporales, aplicamos un enfoque **Point-in-Time (PIT)** estricto mediante la condición `WHERE out_tx.step < t.step`. Esto garantiza que:
* **No hay sesgo de anticipación (*Look-ahead bias*):** El cálculo de cada transacción solo conoce el estado del grafo en el pasado inmediato, ignorando transacciones futuras.
* **No hay fuga de etiqueta (*Target Leakage*):** Nos limitamos a la estructura topológica sin apoyarnos en la variable objetivo `isFraud`.

In [6]:
def calcular_grados_pit(batch_size=40000):
    """
    Calcula in_degree y out_degree históricos para cada transacción
    aplicando Point-in-Time estricto para evitar Data Leakage.
    """
    
    query = """
    CALL apoc.periodic.iterate(
      "MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account) RETURN orig, dest, t",
      
      "CALL {
           WITH orig, t
           MATCH (orig)-[out_tx:TRANSACTION]->()
           WHERE out_tx.step < t.step             
           RETURN count(out_tx) AS out_deg
       }
       CALL {
           WITH dest, t
           MATCH ()-[in_tx:TRANSACTION]->(dest)
           WHERE in_tx.step < t.step             
           RETURN count(in_tx) AS in_deg
       }
       SET t.out_degree_hist = out_deg,
           t.in_degree_hist = in_deg",
      
      {batchSize: $batch_size, parallel: false}
    )
    """
    
    print("Iniciando cálculo de características estructurales en Neo4j...")
    print("Modo: Point-in-Time (PIT) | Protección contra Data Leakage: ACTIVA")
    
    start_time = time.time()
    
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:
            result = session.run(query, batch_size=batch_size)
            summary = result.single()
            
            print("\n--- Resumen del Procesamiento Nativo ---")
            print(f"Total de transacciones analizadas y enriquecidas: {summary['total']}")
            print(f"Lotes (batches) procesados: {summary['batches']}")
            print(f"Operaciones fallidas: {summary['failedOperations']}")
            
            if summary['errorMessages']:
                print(f"Alertas/Errores reportados: {summary['errorMessages']}")
                
    end_time = time.time()
    print(f"\n--- Características creadas con éxito en {round((end_time - start_time)/60, 2)} minutos ---")


calcular_grados_pit()

Iniciando cálculo de características estructurales en Neo4j...
Modo: Point-in-Time (PIT) | Protección contra Data Leakage: ACTIVA

--- Resumen del Procesamiento Nativo ---
Total de transacciones analizadas y enriquecidas: 6362620
Lotes (batches) procesados: 160
Operaciones fallidas: 0

--- Características creadas con éxito en 2.66 minutos ---


## 5. PageRank Temporal por Bloques (Ventanas Deslizantes)

El algoritmo **PageRank** mide la importancia relativa de un nodo en el grafo en función de la estructura de sus conexiones. En detección de fraude, un incremento abrupto en la centralidad de una cuenta suele delatar nodos puente o cuentas "mula" utilizadas para triangular fondos de origen ilícito.

### 🛡️ Estrategia de Ventanas Temporales para Evitar Data Leakage
Calcular PageRank de forma transaccional para cada uno de los 6.3 millones de registros es computacionalmente inviable. Para resolverlo sin caer en sesgo de anticipación (*Look-ahead bias*), implementamos un pipeline por **bloques temporales discretos** (`tamaño_bloque=24`):

1. **Aislamiento del Pasado:** Se proyecta en memoria (`gds.graph.project.cypher`) un subgrafo que contiene *exclusivamente* las transacciones ocurridas hasta el momento límite del bloque (`t.step <= fin_bloque`).
2. **Cálculo de Centralidad:** Se ejecuta PageRank de forma nativa sobre ese estado histórico congelado.
3. **Inyección en el Futuro Inmediato:** Los valores calculados se guardan como atributos históricos (`orig_pagerank_hist`, `dest_pagerank_hist`) únicamente en las transacciones que ocurren en la **ventana de tiempo inmediatamente posterior** (`step > fin_bloque`).
4. **Limpieza eficiente:** Al finalizar todo el ciclo, se eliminan los residuos de los nodos mediante un proceso optimizado por lotes con APOC.

Este diseño metodológico asegura que el modelo predictivo final se entrene con métricas de red que reflejan fielmente el pasado, garantizando la consistencia del modelo en un despliegue en producción real.

In [7]:
def calcular_pagerank_temporal(tamaño_bloque=24):
    STEP_MAXIMO = 743

    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:

            print("=== INICIANDO PIPELINE MAESTRO DE PAGERANK TEMPORAL ===")
            start_global = time.time()

            query_inicializacion = """
            CALL apoc.periodic.iterate(
              "MATCH ()-[t:TRANSACTION]->() RETURN t",
              "SET t.orig_pagerank_hist = 1.0, t.dest_pagerank_hist = 1.0",
              {batchSize: 60000, parallel: false}
            )
            """
            session.run(query_inicializacion)
            print("Inicialización completada con éxito.")

            for inicio_bloque in range(1, STEP_MAXIMO, tamaño_bloque):
                fin_bloque = inicio_bloque + tamaño_bloque - 1
                siguiente_bloque_fin = fin_bloque + tamaño_bloque

                nombre_proyeccion = f"grafo_bloque_{fin_bloque}"

                try:
                    query_proyectar = f"""
                    CALL gds.graph.project.cypher(
                      '{nombre_proyeccion}',
                      'MATCH (a:Account) RETURN id(a) AS id, ["Account"] AS labels',
                      'MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account) WHERE t.step <= {fin_bloque} RETURN id(orig) AS source, id(dest) AS target, "TRANSACTION" AS type'
                    )
                    """
                    session.run(query_proyectar)

                    query_pagerank = f"""
                    CALL gds.pageRank.write(
                      '{nombre_proyeccion}',
                      {{ writeProperty: 'pagerank_temporal' }}
                    )
                    """
                    session.run(query_pagerank)

                    query_congelar = f"""
                    MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account)
                    WHERE t.step > {fin_bloque} AND t.step <= {siguiente_bloque_fin}
                    SET t.orig_pagerank_hist = coalesce(orig.pagerank_temporal, 1.0),
                        t.dest_pagerank_hist = coalesce(dest.pagerank_temporal, 1.0)
                    """
                    session.run(query_congelar)

                    msg = f"Procesado bloque {fin_bloque} -> OK"
                    print(f"\r{msg}", end="", flush=True)

                except Exception as e:
                    msg = f"ERROR bloque {fin_bloque}: {e}"
                    print(f"\r{msg}", end="", flush=True)

                finally:
                    session.run(f"CALL gds.graph.drop('{nombre_proyeccion}', false)")

            query_limpieza_final = """
            CALL apoc.periodic.iterate(
              "MATCH (n:Account) WHERE n.pagerank_temporal IS NOT NULL RETURN n",
              "REMOVE n.pagerank_temporal",
              {batchSize: 60000, parallel: false}
            )
            """
            session.run(query_limpieza_final)

            print()
            print(f"FINALIZADO EN {round((time.time() - start_global)/60, 2)} MINUTOS")

calcular_pagerank_temporal()

=== INICIANDO PIPELINE MAESTRO DE PAGERANK TEMPORAL ===
Inicialización completada con éxito.


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. gds.graph.project.cypher is deprecated. It is replaced by gds.graph.project Cypher projection as an aggregation function.', position=<SummaryInputPosition line=2, column=21, offset=21>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 21, 'line': 2, 'column': 21}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n                    CALL gds.graph.project.cypher(\n                      \'grafo_bloque_24\',\n                      \'MATCH (a:Account) RETURN id(a) AS id, ["Account"] AS labels\',\n                      \'MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account) WHERE t.step <= 24 RET

Procesado bloque 24 -> OK

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('grafo_bloque_24', false)"
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. gds.graph.project.cypher is deprecated. It is replaced by gds.graph.project Cypher projection as an aggregation function.', position=<SummaryInputPosition line=2, column=21, 

Procesado bloque 48 -> OK

...

Procesado bloque 720 -> OK

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('grafo_bloque_720', false)"
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. gds.graph.project.cypher is deprecated. It is replaced by gds.graph.project Cypher projection as an aggregation function.', position=<SummaryInputPosition line=2, column=21,

Procesado bloque 744 -> OK

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('grafo_bloque_744', false)"



FINALIZADO EN 11.96 MINUTOS


## 6. Comunidades de Louvain Temporales (Detección de Fraud Rings)

El algoritmo de **Louvain** maximiza la modularidad del grafo para agrupar dinámicamente los nodos en comunidades densamente conectadas. En entornos financieros, las redes de lavado de dinero o estructuras de colusión (*Fraud Rings*) suelen operar enviando fondos de forma cíclica y cerrada entre cuentas controladas por la misma organización.

### 🛡️ Estrategia de Bloques Temporales para Evitar Data Leakage
Para extraer el comportamiento grupal de la red sin generar sesgo de anticipación (*Look-ahead bias*), estructuramos el pipeline en ventanas deslizantes históricas:

1. **Aislamiento del Grafo Pasado:** Proyectamos en memoria un subgrafo que contiene únicamente las relaciones ocurridas hasta el límite del bloque (`t.step <= fin_bloque`).
2. **Detección de Clusters:** El algoritmo identifica las comunidades estructurales activas en ese periodo y asigna un identificador dinámico a cada nodo (`louvain_id_temp`).
3. **Métrica de Conectividad Grupal (`same_louvain_community_hist`):** Evaluamos las transacciones del bloque futuro inmediato. Si la cuenta origen y destino ya pertenecían a la misma comunidad cerrada en el pasado, la transacción se marca con `1` (Sospecha de colusión interna); de lo contrario, se marca con `0`.

Esta característica dota al modelo de la capacidad de detectar si una transferencia ocurre dentro de un circuito financiero cerrado o si interactúa con el resto de la red legítima.

In [8]:
def calcular_louvain_temporal(tamaño_bloque=24):
    STEP_MAXIMO = 743

    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:

            print("=== INICIANDO PIPELINE MAESTRO DE LOUVAIN TEMPORAL ===")
            start_global = time.time()

            print("[Paso 0.1] Asegurando índice estructural TRANSACTION(step)...")
            session.run("CREATE INDEX tx_step_idx IF NOT EXISTS FOR ()-[r:TRANSACTION]-() ON (r.step);")

            print("[Paso 0.2] Inicializando propiedades base de Louvain en TRANSACTION (Por lotes)...")
            query_inicializacion = """
            CALL apoc.periodic.iterate(
              "MATCH ()-[t:TRANSACTION]->() RETURN t",
              "SET t.same_louvain_community_hist = 0",
              {batchSize: 60000, parallel: false}
            )
            """
            session.run(query_inicializacion)
            print("Inicialización completada con éxito.")

            for inicio_bloque in range(1, STEP_MAXIMO, tamaño_bloque):
                fin_bloque = inicio_bloque + tamaño_bloque - 1
                siguiente_bloque_fin = fin_bloque + tamaño_bloque

                nombre_proyeccion = f"grafo_louvain_bloque_{fin_bloque}"

                try:
                    query_proyectar = f"""
                    CALL gds.graph.project.cypher(
                      '{nombre_proyeccion}',
                      'MATCH (a:Account) RETURN id(a) AS id, ["Account"] AS labels',
                      'MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account) WHERE t.step <= {fin_bloque} RETURN id(orig) AS source, id(dest) AS target, "TRANSACTION" AS type'
                    )
                    """
                    session.run(query_proyectar)

                    query_louvain = f"""
                    CALL gds.louvain.write(
                      '{nombre_proyeccion}',
                      {{ 
                        writeProperty: 'louvain_id_temp',
                        includeIntermediateCommunities: false,
                        concurrency: 4
                      }}
                    )
                    """
                    session.run(query_louvain)

                    query_congelar_lotes = f"""
                    CALL apoc.periodic.iterate(
                      "MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account) WHERE t.step > {fin_bloque} AND t.step <= {siguiente_bloque_fin} RETURN t, orig, dest",
                      "SET t.same_louvain_community_hist = CASE 
                               WHEN orig.louvain_id_temp IS NOT NULL 
                                    AND dest.louvain_id_temp IS NOT NULL 
                                    AND orig.louvain_id_temp = dest.louvain_id_temp THEN 1 
                               ELSE 0 
                           END",
                      {{batchSize: 40000, parallel: false}}
                    )
                    """
                    session.run(query_congelar_lotes)

                    query_limpieza_nodo = """
                    CALL apoc.periodic.iterate(
                      "MATCH (n:Account) WHERE n.louvain_id_temp IS NOT NULL RETURN n",
                      "REMOVE n.louvain_id_temp",
                      {batchSize: 60000, parallel: false}
                    ) YIELD total RETURN total
                    """
                    session.run(query_limpieza_nodo)

                    msg = f"Procesado bloque {fin_bloque} -> OK"
                    print(f"\r{msg}", end="", flush=True)

                except Exception as e:
                    msg = f"ERROR bloque {fin_bloque}: {e}"
                    print(f"\r{msg}", end="", flush=True)

                finally:
                    session.run(f"CALL gds.graph.drop('{nombre_proyeccion}', false)")

            print()
            print(f"FINALIZADO EN {round((time.time() - start_global)/60, 2)} MINUTOS")

calcular_louvain_temporal()

=== INICIANDO PIPELINE MAESTRO DE LOUVAIN TEMPORAL ===
[Paso 0.1] Asegurando índice estructural TRANSACTION(step)...
[Paso 0.2] Inicializando propiedades base de Louvain en TRANSACTION (Por lotes)...
Inicialización completada con éxito.


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. gds.graph.project.cypher is deprecated. It is replaced by gds.graph.project Cypher projection as an aggregation function.', position=<SummaryInputPosition line=2, column=21, offset=21>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 21, 'line': 2, 'column': 21}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n                    CALL gds.graph.project.cypher(\n                      \'grafo_louvain_bloque_24\',\n                      \'MATCH (a:Account) RETURN id(a) AS id, ["Account"] AS labels\',\n                      \'MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account) WHERE t.step <

Procesado bloque 24 -> OK

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('grafo_louvain_bloque_24', false)"
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. gds.graph.project.cypher is deprecated. It is replaced by gds.graph.project Cypher projection as an aggregation function.', position=<SummaryInputPosition line=2, col

Procesado bloque 48 -> OK

...

Procesado bloque 720 -> OK

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('grafo_louvain_bloque_720', false)"
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. gds.graph.project.cypher is deprecated. It is replaced by gds.graph.project Cypher projection as an aggregation function.', position=<SummaryInputPosition line=2, co

Procesado bloque 744 -> OK

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('grafo_louvain_bloque_744', false)"



FINALIZADO EN 23.97 MINUTOS


## 7. Exportación del Dataset Maestro Enriquecido

En esta celda final consolidamos todo el proceso de ingeniería de características (*Feature Engineering*). Extraemos la información de Neo4j y generamos el dataset definitivo que alimentará nuestros modelos de Machine Learning.

Con este paso, el grafo ha cumplido su propósito de transformar conexiones abstractas en variables numéricas predictivas libres de fuga de datos (*Data Leakage*), quedando listo para la fase de modelado.

In [9]:
def exportacion_processed_data(archivo_salida):
    print(f"EXPORTANDO A {archivo_salida}")
    start_time = time.time()

    query = """
    MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account)
    RETURN 
        t.step AS step,
        orig.id AS nameOrig,
        dest.id AS nameDest,
        t.type AS type,
        t.amount AS amount,
        t.oldbalanceOrg AS oldbalanceOrg,
        t.newbalanceOrig AS newbalanceOrig,
        t.oldbalanceDest AS oldbalanceDest,
        t.newbalanceDest AS newbalanceDest,
        t.isFraud AS isFraud,
        t.isFlaggedFraud AS isFlaggedFraud,
        t.in_degree_hist AS in_degree_hist,
        t.out_degree_hist AS out_degree_hist,
        t.orig_pagerank_hist AS orig_pagerank_hist,
        t.dest_pagerank_hist AS dest_pagerank_hist,
        t.same_louvain_community_hist AS same_louvain_community_hist
    """

    columnas = [
        "step", "nameOrig", "nameDest", "type", "amount",
        "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest",
        "isFraud", "isFlaggedFraud",
        "in_degree_hist", "out_degree_hist",
        "orig_pagerank_hist", "dest_pagerank_hist", "same_louvain_community_hist"
    ]

    contador = 0

    with gzip.open(archivo_salida, mode='wt', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(columnas)

        with GraphDatabase.driver(URI, auth=AUTH) as driver:
            with driver.session() as session:
                result = session.run(query)

                for record in result:
                    writer.writerow([record[col] for col in columnas])
                    contador += 1

                    if contador % 50000 == 0:
                        print(f"\rProcesadas {contador} filas...", end="", flush=True)

    print()
    print(f"FINALIZADO EN {round((time.time() - start_time)/60, 2)} MINUTOS | REGISTROS: {contador}")
    print(f"CSV listo en: {archivo_salida}")

exportacion_processed_data("../data/processed/master_dataset_v3.csv.gz")

EXPORTANDO A ../data/processed/master_dataset_v3.csv.gz
Procesadas 6350000 filas...

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `isFlaggedFraud` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=14, column=11, offset=418>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 418, 'line': 14, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account)\n    RETURN \n        t.step AS step,\n        orig.id AS nameOrig,\n        dest.id AS nameDest,\n        t.type AS type,\n        t.amount AS amount,\n        t.oldbalanceOrg AS oldbalanceOrg,\n        t.newbalanceOrig AS newbalanceOrig,\n    


FINALIZADO EN 5.24 MINUTOS | REGISTROS: 6362620
CSV listo en: ../data/processed/master_dataset_v3.csv.gz
